In [1]:
# Test
import os
import numpy as np
import torch
from transformers import AutoTokenizer, Gemma3ForCausalLM
from datasets import load_from_disk

import threading
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import autocast, GradScaler


In [2]:
%load_ext line_profiler

In [3]:
GCP = True

# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# torch.autograd.set_detect_anomaly(True)
TOI = np.fromfile("gemma3_TOI", dtype=np.int32)
if GCP:
    devices = ('cuda:2', 'cuda:3', 'cuda:4')
    model   = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-pt", trust_remote_code=True, cache_dir="/image-generation/imlloja", device_map = devices[0])  
    teacher = Gemma3ForCausalLM.from_pretrained("google/gemma-3-4b-pt", trust_remote_code=True, cache_dir="/image-generation/imlloja", device_map = devices[1])  
    tokenizer   = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt", trust_remote_code=True, cache_dir="/image-generation/imlloja")
    ds = load_from_disk("/image-generation/imlloja/filtered-math")
else:
    devices = ('cuda:0', 'cuda:1', 'cuda:2')
    model   = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-pt", trust_remote_code=True, cache_dir="/nvmes/A4", device_map = devices[0])  
    teacher = Gemma3ForCausalLM.from_pretrained("google/gemma-3-4b-pt", trust_remote_code=True, cache_dir="/nvmes/A4", device_map = devices[1])  
    tokenizer   = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt", trust_remote_code=True, cache_dir="/nvmes/A4")
    teacher_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-pt", trust_remote_code=True, cache_dir="/nvmes/A4")
    ds = load_from_disk("/home/igli/.cache/huggingface/datasets/filtered-math")

VOCABSIZE = tokenizer.vocab_size

neuron_counts = torch.ones(model.lm_head.weight.shape[0], dtype=torch.int32)

configuration = {
    'teacher_model' : teacher,
    'student_model' : model,
    'optimizer' : lambda x : torch.optim.Adam(x, lr=0.001),
    'log_step' : 10,
    'log_path' : 'tensorboards',
    'experiment_name' : 'BASELINEredo_jupyter_smallCritSection',
    'neuron_counts' : neuron_counts,
    'noise_init_fn' : lambda x : torch.nn.init.normal_(x, mean=0, std=0.03),
    'line_length' : 1000,
    'batch_size' : 4,
    'inflated_decoder_sizes' : {
        'cutoffs' : [0,1],
        'counts'  : [   1]
    }
}

for i in range(len(configuration['inflated_decoder_sizes']['cutoffs'])-1):
    counts, cutoffs = configuration['inflated_decoder_sizes']['counts'], configuration['inflated_decoder_sizes']['cutoffs']
    configuration['neuron_counts'][cutoffs[i] : cutoffs[i+1]] = counts[i]

loader = torch.utils.data.DataLoader(ds, batch_size=configuration['batch_size'], shuffle=False)
scaler = GradScaler()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/48 [00:00<?, ?it/s]

In [4]:
from line_profiler import LineProfiler
from functools import wraps

def profile(fn):
    profiler = LineProfiler()

    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = profiler(fn)(*args, **kwargs)
        profiler.print_stats()
        return result
    return wrapper


In [5]:
condition = threading.Condition()
shared_list = []


In [8]:
# @profile
def teacher_loop(dataloader, configuration):
    teacher_model = configuration['teacher_model']
    for i, batch in enumerate(dataloader):
        lines = batch['text']
        batch = tokenizer(lines, max_length=configuration['line_length'], padding=True, truncation=True, return_tensors="pt")
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        with torch.no_grad():
            logits = teacher_model(input_ids.to(teacher_model.device), attention_mask.to(teacher_model.device), return_dict=True, use_cache=False).logits
            teacher_dist = logits.softmax(dim=-1)
            torch.cuda.synchronize()

        with condition:
            while shared_list:
                condition.wait()

            shared_list.append((teacher_dist, input_ids, attention_mask, i))
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            condition.notify()

        if i > 30:
            break

    with condition:
        shared_list.append(None)
        condition.notify()

# @profile
def student_loop(configuration):
    def normalize(x): return x / x.sum(dim=-1, keepdim=True)
    
    writer = SummaryWriter(f"{configuration['log_path']}/{configuration['experiment_name']}")
    student_model = configuration['student_model']
    decoder_device = devices[2]
    counts = configuration['neuron_counts']
    cum_neurons = torch.nn.functional.pad(torch.cumsum(counts, dim=-1), (1,0), mode='constant', value=0)

    with torch.no_grad():
        data = torch.repeat_interleave(student_model.lm_head.weight.data, counts.to(devices[0]), dim=0).to(device=decoder_device, dtype=torch.float32)
    inflated_decoder = torch.nn.Linear(data.shape[1], data.shape[0], bias=False).to(decoder_device)
    inflated_decoder.weight.data.copy_(data)

    optimizer = configuration['optimizer'](inflated_decoder.parameters())

    while True:
        with condition:
            while not shared_list:
                condition.wait()

            data = shared_list.pop(0)
            if data is None:
                break
            teacher_dist, input_ids, attention_masks, i = data
            condition.notify()

        optimizer.zero_grad()
        torch.cuda.synchronize()

        with torch.no_grad():
            backbone_out = student_model.model(input_ids.to(student_model.device), attention_masks.to(student_model.device), return_dict=True, use_cache=False)
        inflated_input = backbone_out.last_hidden_state.to(dtype=torch.float32, device=decoder_device).contiguous()
        torch.cuda.synchronize()

        inflated_logits = inflated_decoder(inflated_input)
        torch.cuda.synchronize()

        with torch.no_grad():
            stabilizers = inflated_logits.max(dim=-1, keepdims=True).values.detach()
        inflated_logits = inflated_logits - stabilizers
        torch.cuda.synchronize()

        acc_logits = torch.exp(inflated_logits).cumsum(dim=-1)
        acc_logits = torch.nn.functional.pad(acc_logits, (1,0), mode='constant', value=0)
        distribution = acc_logits[..., cum_neurons[1:]] - acc_logits[..., cum_neurons[:-1]]
        distribution = distribution / acc_logits[..., -1:]
        torch.cuda.synchronize()

        tempi = torch.clamp(distribution, min=1e-12)
        tempt = torch.clamp(teacher_dist.to(tempi.device), min=1e-12)
        kl = torch.nn.functional.kl_div(normalize(tempi)[...,:VOCABSIZE].log(), normalize(tempt)[...,:VOCABSIZE])

        kl.backward()
        torch.cuda.synchronize()
        torch.nn.utils.clip_grad_value_(inflated_decoder.parameters(), clip_value=1e-5)
        optimizer.step()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

        if i % configuration['log_step'] == 0:
            original_logits = student_model.lm_head(backbone_out.last_hidden_state).softmax(dim=-1).detach()
            original_loss = torch.nn.functional.kl_div(normalize(original_logits)[...,:VOCABSIZE].log().to(tempt.device), normalize(tempt)[...,:VOCABSIZE])
            writer.add_scalar('original loss', original_loss, i)
            writer.add_scalar('improvement', original_loss - kl.item(), i)

        del teacher_dist, input_ids, attention_masks, backbone_out
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

def main(configuration):
    t1 = threading.Thread(target=teacher_loop, args=(loader, configuration))
    t2 = threading.Thread(target=student_loop, args=(configuration,))
    t1.start()
    t2.start()
    t1.join()
    t2.join()

In [9]:
configuration['log_path'] = "tensorboards/BASELINE_mergedCodeInSingleCell"
main(configuration)

In [5]:
class InflatedDecoder(torch.nn.Module):
    def __init__(self,old_decoder, counts, mode='fresh', device=devices[2], noise_init_fn = None):
        super().__init__()
        self.device = device
        with torch.no_grad():
            if noise_init_fn:
                noise = noise_init_fn(torch.empty(counts.sum().item(), old_decoder.weight.data.size(1))).to(device)*0
            if mode == "centered":
                data = torch.repeat_interleave(old_decoder.weight.data, counts.to(devices[0]), dim=0).to(device=device, dtype=torch.float32)
            if mode == "fresh":
                data = torch.empty((counts.sum().item(), old_decoder.weight.data.size(1)), dtype=torch.float32)
                torch.nn.init.kaiming_uniform_(data, a=torch.math.sqrt(5))
            if mode == "noise":
                data = torch.empty((counts.sum().item(), old_decoder.weight.data.size(1)), dtype=torch.float32)
                torch.nn.init.zeros_(data)
            if noise_init_fn:
                data += noise
            data = data.to(device)
        self.weights = torch.nn.Parameter(data)

        self.bias = None 
    def forward(self, x):
        x = x.to(dtype=torch.float32, device=self.device).contiguous()

        return torch.nn.functional.linear(x, self.weights, self.bias)

def loss_fn(input_dist, target_dist):
    def my_kl(x, y):
        ylog, xlog = y.log(), x.log()
        diff = ylog - xlog
        return (y * diff).mean()

    def normalize(x):
        return x / x.sum(dim=-1, keepdim=True)
    input_dist = torch.clamp(input_dist, min=1e-12)
    target_dist = torch.clamp(target_dist, min=1e-12)

    target_dist = target_dist.to(input_dist.device)
    tempi, tempt = normalize(input_dist[...,:VOCABSIZE]), normalize(target_dist[...,:VOCABSIZE])
    out = torch.nn.functional.kl_div(tempi.log(), tempt)

    return out

def get_teacher_distribution(model, input_ids, attention_mask):
    with torch.no_grad():
        out = model.forward(input_ids, attention_mask, return_dict=True, use_cache=False, output_hidden_states=False)
        torch.cuda.synchronize()
        return out['logits'].softmax(dim=-1)

@profile
def get_student_distribution(student_model, inflated_decoder, cummulative_neuron_counts, input_ids, attention_mask):
    with torch.no_grad():
        backbone_out = student_model.model(input_ids, attention_mask, return_dict=True,use_cache=False)
        # torch.cuda.synchronize(device=backbone_out.last_hidden_state.device)

    # with autocast(dtype=torch.float32):
    # with torch.cuda.device('cuda:4'):
    inflated_input = backbone_out.last_hidden_state.to(dtype=torch.float32, device=inflated_decoder.device).contiguous()
    torch.cuda.synchronize()
    try:
        inflated_logits = inflated_decoder(inflated_input)
        torch.cuda.synchronize()
    except RuntimeError as e:
        raise

    with torch.no_grad():
        stabilizers = inflated_logits.max(dim=-1, keepdims=True).values.detach()
        torch.cuda.synchronize()
    inflated_logits = inflated_logits - stabilizers
    torch.cuda.synchronize()
    accummulated_logits = torch.exp(inflated_logits).cumsum(dim=-1)
    torch.cuda.synchronize()

    accummulated_logits = torch.nn.functional.pad(accummulated_logits, (1,0), mode='constant', value=0)

    distribution = accummulated_logits[..., cummulative_neuron_counts[1:]] - accummulated_logits[..., cummulative_neuron_counts[:-1]]
    torch.cuda.synchronize()
    distribution = distribution / accummulated_logits[..., -1:]
    torch.cuda.synchronize()

    del accummulated_logits, stabilizers
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    return backbone_out.last_hidden_state, distribution

@profile
def learn_one_step(student_model, optimizer, distributions, inflated_decoder, cummulative_neuron_counts,input_ids,attention_mask, student_forward_results, teacher_dist):

    optimizer.zero_grad()
    torch.cuda.synchronize()
    # with autocast(dtype=torch.float32):
    hidden_states, student_distribution = get_student_distribution(student_model, inflated_decoder, cummulative_neuron_counts, input_ids, attention_mask)
    torch.cuda.synchronize()

    loss = loss_fn(student_distribution, teacher_dist)
    loss.backward()
    torch.cuda.synchronize()
    torch.nn.utils.clip_grad_value_(inflated_decoder.parameters(), clip_value=1e-5)
    optimizer.step()
    torch.cuda.synchronize()

    torch.cuda.empty_cache()
    # torch.cuda.synchronize()
    student_forward_results.append((hidden_states, loss.item()))

def teacher_prepare(model, input_ids, attention_mask):
    return get_teacher_distribution(model, input_ids.to(model.device), attention_mask.to(model.device))

@profile
def teacher_loop(dataloader, configuration):
    teacher_model = configuration['teacher_model']
    for i,batch in enumerate(dataloader):
        lines = batch['text']
        batch = tokenizer(lines, max_length=configuration['line_length'], padding=True, truncation=True, return_tensors="pt")
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']

        with condition:
            while shared_list:
            teacher_dist = teacher_prepare(teacher_model, input_ids, attention_mask)
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
                condition.wait()

            shared_list.append((teacher_dist, input_ids, attention_mask, i))

            condition.notify()
        if i > 30:
            break
    with condition:
        shared_list.append(None)
        condition.notify()

@profile
def student_loop(configuration):
    writer = SummaryWriter(f"{configuration['log_path']}/{configuration['experiment_name']}")
    student_model = configuration['student_model']
    cum_neurons = torch.nn.functional.pad(torch.cumsum(configuration['neuron_counts'], dim=-1), (1,0), mode='constant', value=0)
    decoder_device = devices[2]
    inflated_decoder = InflatedDecoder(student_model.lm_head, configuration['neuron_counts'], mode="centered", device=decoder_device, noise_init_fn=configuration['noise_init_fn']).to(decoder_device)

    optimizer = configuration['optimizer'](inflated_decoder.parameters())

    while True:
        with condition:
            while not shared_list:
                condition.wait()

            data = shared_list.pop(0)

            if data is None:
                break
            teacher_dist, input_ids, attention_masks, i = data
            condition.notify()

        student_results = []
        learn_one_step(student_model, optimizer, (None), inflated_decoder, cum_neurons, input_ids.to(student_model.device), attention_masks.to(student_model.device), student_results, teacher_dist)
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        if i % configuration['log_step'] == 0:

            student_results = student_results.pop(0)
            original_distributions = student_model.lm_head(student_results[0]).softmax(dim=-1).detach()
            torch.cuda.synchronize()

            original_loss = loss_fn(original_distributions, teacher_dist)
            inflated_decoder_loss = student_results[1]

            writer.add_scalar('original loss', original_loss, i)
            writer.add_scalar('improvement', original_loss - inflated_decoder_loss, i)
        del teacher_dist, input_ids, attention_masks, student_results
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

condition = threading.Condition()
shared_list = []
def main(configuration):
    t1 = threading.Thread(target=teacher_loop, args=(loader, configuration))
    t2 = threading.Thread(target=student_loop, args=(configuration,))
    t1.start()
    t2.start()
    t1.join()
    t2.join()


In [6]:
main(configuration)

Timer unit: 1e-09 s

Total time: 0.551471 s
File: /tmp/ipykernel_3030169/1871415812.py
Function: get_student_distribution at line 50

Line #      Hits         Time  Per Hit   % Time  Line Contents
    50                                           @profile
    51                                           def get_student_distribution(student_model, inflated_decoder, cummulative_neuron_counts, input_ids, attention_mask):
    52         2      20779.0  10389.5      0.0      with torch.no_grad():
    53         1  104486329.0    1e+08     18.9          backbone_out = student_model.model(input_ids, attention_mask, return_dict=True,use_cache=False)
    54                                                   # torch.cuda.synchronize(device=backbone_out.last_hidden_state.device)
    55                                           
    56                                               # with autocast(dtype=torch.float32):
    57                                               # with torch.cuda.device('cud

/image-generation/imlloja/igli/lib/python3.11/site-packages/torch/nn/functional.py:2976: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  warnings.warn(


Timer unit: 1e-09 s

Total time: 0.000500523 s
File: /tmp/ipykernel_3030169/1871415812.py
Function: learn_one_step at line 86

Line #      Hits         Time  Per Hit   % Time  Line Contents
    86                                           @profile
    87                                           def learn_one_step(student_model, optimizer, distributions, inflated_decoder, cummulative_neuron_counts,input_ids,attention_mask, student_forward_results, teacher_dist):
    88                                           
    89         1     410830.0 410830.0     82.1      optimizer.zero_grad()
    90         1      89693.0  89693.0     17.9      torch.cuda.synchronize()
    91                                               # with autocast(dtype=torch.float32):
    92                                               hidden_states, student_distribution = get_student_distribution(student_model, inflated_decoder, cummulative_neuron_counts, input_ids, attention_mask)
    93                              

In [7]:
1+1

2

In [ ]:
neuron_counts = torch.ones(VOCABSIZE, dtype=torch.int32)
neuron_counts[TOI[    : 200]] = 500
neuron_counts[TOI[ 200:1000]] = 100
neuron_counts[TOI[1000:5000]] = 50
configuration = {
    'teacher_model' : teacher,
    'student_model' : model,
    'optimizer' : lambda x : torch.optim.Adam(x, lr=0.001),
    'log_step' : 10,
    'log_path' : 'gcptest',
    'experiment_name' : 'progressive_clustering/noisy_centered_looseClamp',
    'neuron_counts' : neuron_counts,
    'noise_init_fn' : lambda x : torch.nn.init.normal_(x, mean=0, std=0.03)
}

In [ ]:
teacher_model = configuration['teacher_model']
for i,batch in enumerate(loader):
    lines = batch['text']
    batch = tokenizer(lines, max_length=512, padding=True, truncation=True, return_tensors="pt")
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    teacher_dist = teacher_prepare(teacher_model, input_ids, attention_mask)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    shared_list.append((teacher_dist, input_ids, attention_mask, i))
    break

In [ ]:
student_model = configuration['student_model']
cum_neurons = torch.nn.functional.pad(torch.cumsum(configuration['neuron_counts'], dim=-1), (1,0), mode='constant', value=0)
inflated_decoder = InflatedDecoder(student_model.lm_head, configuration['neuron_counts'], mode="centered", noise_init_fn=configuration['noise_init_fn']).to('cuda:2')
# print("Finished inflating")
optimizer = configuration['optimizer'](inflated_decoder.parameters())



data = shared_list.pop(0)
if data is None:
    print("oops")
teacher_dist, input_ids, attention_masks, i = data


In [ ]:
from collections import Counter
token_counter = Counter()
for batch in loader:
    line = batch['text'][0]
    tokens = tokenizer(line).input_ids
    token_counter.update(tokens)

In [ ]:
most_common = token_counter.most_common(10000)
IDS = [token_id for token_id, _ in most_common]
import numpy as np
IDS_np = np.array(IDS, dtype=np.int32)
IDS_np.tofile("gemma3_TOI")

In [ ]:
(loaded_toi == IDS_np).all()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(list(map(lambda x : x[1], most_common)))
plt.yscale('log')
plt.xscale('log')